# 可选实验：特征工程与多项式回归

![](./图片/C1_W2_Lab07_FeatureEngLecture.PNG)


## 目标
在本实验中你将：
- 探索特征工程与多项式回归，它们让你能利用线性回归的机制来拟合非常复杂、甚至高度非线性的函数。


## 工具
你将使用之前实验中开发的函数，以及 matplotlib 和 NumPy。 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from lab_utils_multi import zscore_normalize_features, run_gradient_descent_feng
np.set_printoptions(precision=2)  # 降低 numpy 数组的显示精度

<a name='FeatureEng'></a>
# 特征工程与多项式回归概述

开箱即用时，线性回归提供了构建如下形式模型的手段：
$$f_{\mathbf{w},b} = w_0x_0 + w_1x_1+ ... + w_{n-1}x_{n-1} + b \tag{1}$$ 
如果你的特征/数据是非线性的，或者是特征的组合呢？例如，房价与居住面积往往不是线性的，而且对很小或很大的房子会有折价，形成上图所示的曲线。我们如何利用线性回归的机制来拟合这条曲线？回想一下，我们手上的“机制”是：调整 (1) 中的参数 $\mathbf{w}$、$\mathbf{b}$，使方程“拟合”训练数据。然而，无论怎样调整 (1) 中的 $\mathbf{w}$、$\mathbf{b}$，都无法拟合一条非线性曲线。


<a name='PolynomialFeatures'></a>
## 多项式特征

上面我们考虑的是数据非线性的场景。让我们试着用目前掌握的知识来拟合一条非线性曲线。从一个简单的二次函数开始：$y = 1+x^2$

我们使用的例程你都很熟悉。它们可以在 lab_utils.py 文件中查看。我们将用到 [`np.c_[..]`](https://numpy.org/doc/stable/reference/generated/numpy.c_.html)，这是 NumPy 中沿列边界拼接的例程。

In [ ]:
# 创建目标数据
x = np.arange(0, 20, 1)
y = 1 + x**2
X = x.reshape(-1, 1)

model_w,model_b = run_gradient_descent_feng(X,y,iterations=1000, alpha = 1e-2)

plt.scatter(x, y, marker='x', c='r', label="Actual Value"); plt.title("no feature engineering")
plt.plot(x,X@model_w + model_b, label="Predicted Value");  plt.xlabel("X"); plt.ylabel("y"); plt.legend(); plt.show()

不出所料，拟合得不太好。需要的是类似 $y= w_0x_0^2 + b$ 的形式，也就是一个**多项式特征**。
为此，你可以修改*输入数据*，*工程化*出所需的特征。如果把原始数据替换为对 $x$ 值取平方的版本，就能实现 $y= w_0x_0^2 + b$。让我们试试。把下面的 `X` 换成 `X**2`：

In [ ]:
# 创建目标数据
x = np.arange(0, 20, 1)
y = 1 + x**2

# 构造特征
X = x**2      #<-- 新增的构造特征

In [ ]:
X = X.reshape(-1, 1)  #X 应当是二维矩阵
model_w,model_b = run_gradient_descent_feng(X, y, iterations=10000, alpha = 1e-5)

plt.scatter(x, y, marker='x', c='r', label="Actual Value"); plt.title("Added x**2 feature")
plt.plot(x, np.dot(X,model_w) + model_b, label="Predicted Value"); plt.xlabel("x"); plt.ylabel("y"); plt.legend(); plt.show()

太好了！几乎完美拟合。注意图上打印的 $\mathbf{w}$ 和 b 值：`w,b found by gradient descent: w: [1.], b: 0.0490`。梯度下降把我们初始的 $\mathbf{w},b $ 调整为 (1.0,0.049)，即模型 $y=1*x_0^2+0.049$，非常接近我们的目标 $y=1*x_0^2+1$。如果运行更久，会匹配得更好。 

### 选择特征
<a name='GDF'></a>
上面我们知道需要 $x^2$ 项。但需要哪些特征并不总是显而易见的。可以加入各种候选特征，试着找出最有用的那些。例如，如果我们改试：$y=w_0x_0 + w_1x_1^2 + w_2x_2^3+b$ 呢？

运行接下来的单元格。 

In [ ]:
# 创建目标数据
x = np.arange(0, 20, 1)
y = x**2

# 构造特征 .
X = np.c_[x, x**2, x**3]   #<-- 新增的构造特征

In [ ]:
model_w,model_b = run_gradient_descent_feng(X, y, iterations=10000, alpha=1e-7)

plt.scatter(x, y, marker='x', c='r', label="Actual Value"); plt.title("x, x**2, x**3 features")
plt.plot(x, X@model_w + model_b, label="Predicted Value"); plt.xlabel("x"); plt.ylabel("y"); plt.legend(); plt.show()

注意 $\mathbf{w}$ 的值 `[0.08 0.54 0.03]`，b 为 `0.0106`。这意味着拟合/训练后的模型是：
$$ 0.08x + 0.54x^2 + 0.03x^3 + 0.0106 $$
梯度下降通过相对其他项增大 $w_1$ 项，强调了最适合 $x^2$ 数据的那部分数据。如果你运行非常久，它会继续降低其他项的影响。 
>梯度下降通过强调相应参数，替我们挑出了“正确”的特征

让我们回顾这个思路：
- 一开始，特征被重新缩放，使它们彼此可比
- 权重值越小意味着该特征越不重要/越不正确，极端情况下当权重变为零或非常接近零时，该特征对拟合模型没有用处。
- 上面拟合后，与 $x^2$ 特征相关的权重远大于 $x$ 或 $x^3$ 的权重，因为它在拟合数据中最有用。 

### 另一种视角
上面，多项式特征是根据它们与目标数据的匹配程度来选择的。另一种理解方式是：创建新特征之后，我们用的仍然是线性回归。既然如此，最好的特征将是相对目标呈线性的特征。用一个例子最容易理解。 

In [ ]:
# 创建目标数据
x = np.arange(0, 20, 1)
y = x**2

# 构造特征 .
X = np.c_[x, x**2, x**3]   #<-- 新增的构造特征
X_features = ['x','x^2','x^3']

In [ ]:
fig,ax=plt.subplots(1, 3, figsize=(12, 3), sharey=True)
for i in range(len(ax)):
    ax[i].scatter(X[:,i],y)
    ax[i].set_xlabel(X_features[i])
ax[0].set_ylabel("y")
plt.show()

上面可以清楚地看到，$x^2$ 特征与目标值 $y$ 的关系是线性的。线性回归随后就能轻松地用该特征生成模型。

### 特征缩放
如上一个实验所述，如果数据集中的特征尺度差异很大，应做特征缩放以加速梯度下降。在上面的例子中有 $x$、$x^2$ 和 $x^3$，它们的尺度天然差异很大。让我们对本例应用 z 分数归一化。

In [ ]:
# 创建目标数据
x = np.arange(0,20,1)
X = np.c_[x, x**2, x**3]
print(f"Peak to Peak range by column in Raw        X:{np.ptp(X,axis=0)}")

# 加入均值归一化
X = zscore_normalize_features(X)     
print(f"Peak to Peak range by column in Normalized X:{np.ptp(X,axis=0)}")

现在我们可以用一个更激进的 alpha 取值再试一次：

In [ ]:
x = np.arange(0,20,1)
y = x**2

X = np.c_[x, x**2, x**3]
X = zscore_normalize_features(X) 

model_w, model_b = run_gradient_descent_feng(X, y, iterations=100000, alpha=1e-1)

plt.scatter(x, y, marker='x', c='r', label="Actual Value"); plt.title("Normalized x x**2, x**3 feature")
plt.plot(x,X@model_w + model_b, label="Predicted Value"); plt.xlabel("x"); plt.ylabel("y"); plt.legend(); plt.show()

特征缩放让它收敛得快得多。   
再次注意 $\mathbf{w}$ 的取值。$w_1$ 项即 $x^2$ 项被强调得最多。梯度下降几乎消除了 $x^3$ 项。

### 复杂函数
借助特征工程，即使相当复杂的函数也能被建模：

In [ ]:
x = np.arange(0,20,1)
y = np.cos(x/2)

X = np.c_[x, x**2, x**3,x**4, x**5, x**6, x**7, x**8, x**9, x**10, x**11, x**12, x**13]
X = zscore_normalize_features(X) 

model_w,model_b = run_gradient_descent_feng(X, y, iterations=1000000, alpha = 1e-1)

plt.scatter(x, y, marker='x', c='r', label="Actual Value"); plt.title("Normalized x x**2, x**3 feature")
plt.plot(x,X@model_w + model_b, label="Predicted Value"); plt.xlabel("x"); plt.ylabel("y"); plt.legend(); plt.show()



## 恭喜！
在本实验中你：
- 了解了线性回归如何借助特征工程对复杂的、甚至高度非线性的函数建模
- 认识到做特征工程时应用特征缩放很重要